# Stage 1: build the dataset

Streams a month of the Lichess open database, keeps games where both players sit in the configured rating band, samples opening positions, annotates each one with Stockfish (evaluation, best line, motifs) and writes the training files.

- **Runtime type**: CPU is fine for this notebook. No GPU needed.
- **Cost**: nothing. Colab free tier plus your Google Drive for storage.
- **Time**: roughly two to three hours, almost all of it engine annotation.
- **Interruptions are fine**: annotation caches every result to Drive, so if the session recycles you just run all cells again and it resumes where it stopped.


In [ ]:
REPO_URL = "https://github.com/ariyan-tipu/opening-theory-engine.git"

# A plain install rather than -e: editable installs only register when the
# interpreter starts, and a live Colab kernel has already started.
!rm -rf /content/opening-theory-engine
!git clone --depth 1 {REPO_URL} /content/opening-theory-engine
%pip install -q /content/opening-theory-engine
print("install done")

In [ ]:
# Work on Drive so every artefact survives session recycling.
from google.colab import drive

drive.mount("/content/drive")

import os

WORKDIR = "/content/drive/MyDrive/opening-theory-engine"
os.makedirs(WORKDIR, exist_ok=True)
%cd {WORKDIR}

In [ ]:
# Pinned Stockfish 18, checksum verified by the fetch tool.
import os

from opening_theory_engine.engine.fetch import fetch

engine_path = fetch()
os.environ["STOCKFISH_PATH"] = str(engine_path)
print("engine at", engine_path)

In [ ]:
!ote engine check --config /content/opening-theory-engine/configs/colab-t4.yaml

In [ ]:
# The long cell. Safe to re-run after any interruption. Quiet mode keeps the
# browser tab light; progress is visible as the cache file grows on Drive.
!ote data build --config /content/opening-theory-engine/configs/colab-t4.yaml --quiet

In [ ]:
# What did we end up with?
print(open("data/processed/dataset_card.md", encoding="utf-8").read())

Done. The files under `data/processed/` on your Drive feed the next notebook, `02_train_sft.ipynb`.
